# Methane plume matched-filter and flux lab
This lab uses synthetic enhancement fields. It starts after retrieval and does not operate on raw satellite radiance.

In [ ]:
import sys
sys.path.insert(0, '../implementations/python/methane-emissions')
import numpy as np
import matplotlib.pyplot as plt
from methane_emissions import gaussian_plume_template, matched_filter_plume, ime_emission_rate, cross_sectional_flux
rng=np.random.default_rng(7)
image=rng.normal(0,.2,(81,91))
template=gaussian_plume_template((15,25),2,5,.3)
image[33:48,34:59]+=4*template
result=matched_filter_plume(image,[template],.2,threshold=5)
plt.figure(figsize=(12,4));plt.subplot(1,2,1);plt.imshow(image);plt.title('Synthetic enhancement');plt.subplot(1,2,2);plt.imshow(result['score']);plt.title('Matched-filter score');plt.show()


In [ ]:
# Dimensional examples: IME [kg], wind [m/s], length [m] -> kg/s
ime_emission_rate(100,5,20)
concentration=np.array([1.,2.,3.]) # kg/m3
normal_wind=np.array([2.,2.,-1.]) # m/s
segment_width=np.array([1.,.5,2.]) # m
cross_sectional_flux(concentration,normal_wind,segment_width)


## Interpretation and limits
A detection score is not an emissions estimate. IME and cross-sectional flux use different geometric approximations and require independently defensible concentration, wind and length/width inputs. The white-noise matched-filter assumption is especially fragile for spatially correlated backgrounds.

## Robust local background removal
A rolling median estimates slowly varying background; local MAD standardizes residual enhancements. The window must be chosen relative to plume width and background scale.

In [ ]:
from methane_emissions import robust_local_background
yy, xx = np.mgrid[:60, :80]
background_field = 0.02*xx + 0.01*yy
background_field[20:26, 20:48] += 2.5
corrected = robust_local_background(background_field, window_size=13, scale_floor=0.02)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.imshow(background_field); plt.title('Input with gradient')
plt.subplot(1,2,2); plt.imshow(corrected['standardized_enhancement']); plt.title('Local MAD-standardized enhancement')
plt.show()

## Hysteresis plume segmentation
Low-threshold connected regions are retained only when they contain a high-threshold seed. This reduces isolated weak-noise detections while preserving connected plume tails.

In [ ]:
from methane_emissions import hysteresis_plume_segmentation
synthetic = np.zeros((60,80))
synthetic[20:28,15:48] = 1.4
synthetic[22:26,20:30] = 4.0
synthetic[42:46,60:65] = 1.3  # isolated weak patch
seg = hysteresis_plume_segmentation(synthetic, low_threshold=1.0, high_threshold=3.0, min_area=5)
print('Detected plume objects:', seg['objects'])
plt.imshow(seg['labels']); plt.title('Hysteresis plume segmentation'); plt.show()